In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

OUTPUT_DIR = PROJECT_ROOT / "outputs"

CLEAN_DATA_PATH = OUTPUT_DIR / "APL_Logistics_cleaned.csv"

In [3]:
df = pd.read_csv(
    CLEAN_DATA_PATH,
    encoding="latin1"
)

In [4]:
print("Rows:",df.shape[0])
print("Columns:",df.shape[1])

Rows: 180519
Columns: 40


In [6]:
TARGET = "Late_delivery_risk"
df[TARGET].value_counts()

Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

In [13]:
LEAKAGE_COLUMNS = [
    "Late_delivery_risk",
    "Days for shipping (real)",
    "Delivery Status",
    "Order Status"
]

In [16]:
PII_COLUMNS = [
    "Customer Fname",
    "Customer Lname",
    "Customer Street",
    "Customer Id",
    "Order Customer Id",
    "Customer Zipcode"
]

In [17]:
missing_leakage = [
    col for col in LEAKAGE_COLUMNS
    if col not in df.columns
]

missing_pii = [
    col for col in PII_COLUMNS
    if col not in df.columns
]

print("Missing leakage columns:",missing_leakage)
print("Missing PII columns:",missing_pii)

Missing leakage columns: []
Missing PII columns: []


In [22]:
df["discount_amount_per_unit"] = (
    df["Order Item Discount"] 
    / df["Order Item Quantity"].clip(lower=1)
)

In [25]:
df["sales_per_quantity"] = (
    df["Sales"] 
    / df["Order Item Quantity"].clip(lower=1)
)

In [27]:
df["profit_per_quantity"] = (
    df["Order Profit Per Order"]
    / df["Order Item Quantity"].clip(lower=1)
)

In [29]:
df["price_discount_interaction"] = (
    df["Order Item Product Price"]
    * df["Order Item Discount Rate"]
)

In [31]:
df["scheduled_days_bucket"] = (
    df["Days for shipment (scheduled)"]
    .astype(str)
)

In [32]:
df["geo_market_region"] = (
    df["Market"].astype(str)
    + " | "
    + df["Order Region"].astype(str)
)

In [34]:
new_features = [
    "discount_amount_per_unit",
    "sales_per_quantity",
    "profit_per_quantity",
    "price_discount_interaction",
    "scheduled_days_bucket",
    "geo_market_region"
]

df[new_features].head()

,discount_amount_per_unit,sales_per_quantity,profit_per_quantity,price_discount_interaction,scheduled_days_bucket,geo_market_region
0,5.500,99.99,31.938,5.9994,4,Pacific Asia | South Asia
1,6.398,39.99,9.742,6.3984,4,LATAM | Central America
2,18.000,199.99,87.360,17.9991,4,LATAM | Central America
3,24.000,199.99,-41.890,23.9988,4,USCA | East of USA
4,10.000,50.00,10.000,10.0000,4,USCA | East of USA


In [35]:
np.isinf(
    df.select_dtypes(include=np.number)
).sum().sum()

np.int64(0)

In [36]:
df[new_features].isnull().sum()

discount_amount_per_unit      0
sales_per_quantity            0
profit_per_quantity           0
price_discount_interaction    0
scheduled_days_bucket         0
geo_market_region             0
dtype: int64

In [37]:
DROP_COLUMNS = LEAKAGE_COLUMNS + PII_COLUMNS

In [40]:
ml_df = df.drop(
    columns = DROP_COLUMNS,
    errors = "ignore"
).copy()

In [41]:
ml_df.shape

(180519, 36)

In [42]:
x = ml_df.drop(
    columns=[TARGET],
    errors="ignore"
)

y=df[TARGET].astype(int)

In [43]:
print("x shape:",x.shape)
print("y shape:",y.shape)

x shape: (180519, 36)
y shape: (180519,)


In [44]:
TARGET in x.columns

False

In [45]:
remaining_leakage = [
    col for col in LEAKAGE_COLUMNS
    if col in x.columns
]

remaining_leakage

[]

In [47]:
remaining_pii = [
    col for col in PII_COLUMNS
    if col in x.columns
]

remaining_pii

[]